# Autoencoder Video Compression
This notebook runs the full pipeline: setup, data prep, training, evaluation, and visualization.

## 1. Setup

In [ ]:
import os
import glob
import zipfile

# Find and unzip the first .zip file in the current directory
zip_files = glob.glob('*.zip')
if zip_files:
    print(f'Found zip: {zip_files[0]}')
    with zipfile.ZipFile(zip_files[0], 'r') as z:
        z.extractall('.')
else:
    print('No .zip file found — skipping extraction.')

# cd into the extracted folder if it created one
extracted = [d for d in os.listdir('.') if os.path.isdir(d) and 'Autoencoders' in d]
if extracted:
    os.chdir(extracted[0])

print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Dataset
Download the VIRAT aerial videos (~4.9 GB) from the Kitware Data Portal.

In [ ]:
!python scripts/download_dataset.py

## 3. Extract Frames

In [ ]:
!python scripts/extract_frames.py

## 4. Train

In [ ]:
!python ml/train.py

In [ ]:
# Find the latest run folder for downstream scripts
import glob
runs = sorted(glob.glob('ml/models/saved/*'))
RUN = runs[-1]
print(f'Latest run: {RUN}')
print(f'Contents: {os.listdir(RUN)}')

## 5. Plot Training Curves

In [ ]:
!python scripts/plot_metrics.py --run {RUN}

In [ ]:
from IPython.display import Image, display
img_path = os.path.join(RUN, 'loss_curves.png')
print(f'Looking for: {os.path.abspath(img_path)}')
print(f'Exists: {os.path.exists(img_path)}')
print(f'Notebook cwd: {os.getcwd()}')
display(Image(filename=img_path))

## 6. Evaluate on Test Set

In [ ]:
!python scripts/evaluate.py --run {RUN}

## 7. Visual Inspection

In [ ]:
!python scripts/visualize.py --run {RUN}

In [ ]:
display(Image(filename=f'{RUN}/visual_inspection.png'))

## 8. Compression & Latent Space Analysis

In [ ]:
!python scripts/analysis.py --run {RUN}

In [ ]:
display(Image(filename=f'{RUN}/latent_space_pca_2D.png'))
display(Image(filename=f'{RUN}/latent_space_pca_3D.png'))

## 9. Download Model (optional)

In [ ]:
# Uncomment to download from Colab
# from google.colab import files
# files.download(f'{RUN}/best_model.pth')
# files.download(f'{RUN}/metrics.csv')